# Generative AI Mini-Project 1 — Single Image Super-Resolution (x4)
**Master 2 I2A — Residual-Attention U-Net Convolutional Autoencoder (RA-UNet-AE)**

## Design summary (use this for the 2-page report)

**Architecture — a convolutional encoder / bottleneck / decoder autoencoder, fully convolutional**
- **Encoder (LR space):** 3x3 stem conv -> residual channel-attention groups @64x64 -> strided conv -> groups @32x32 -> strided conv -> **latent bottleneck @16x16** (residual blocks with channel + spatial attention).
- **Decoder:** `PixelShuffle` up-sampling 16->32->64 with **encoder-decoder skip connections** (concat + 1x1 fuse) and residual attention groups at each scale, plus a feature-level global skip from the stem.
- **x4 reconstruction head:** two `PixelShuffle(2)` stages 64->128->256 (ICNR-initialised, no checkerboard artefacts) with a refinement residual group at 128x128.
- **Bicubic residual learning:** `output = bicubic_x4(LR) + network(LR)`; the last conv is zero-initialised, so training starts exactly at the bicubic baseline and the network only learns the missing high-frequency detail.
- Building blocks: residual block (conv-ReLU-conv, residual scaling 0.2) + **channel attention** (squeeze-and-excitation) + optional **spatial attention**. No BatchNorm by default (it hurts PSNR in SR — flag `USE_BN` is available).

**Training decisions (PSNR is the leaderboard metric, so everything optimises pixel fidelity)**
1. Pixel losses only: Charbonnier (smooth L1) for the main stage, MSE for the final fine-tune stage (it directly optimises PSNR). No perceptual/GAN loss (they lower PSNR).
2. **Two stages:** (1) joint training on the provided data + optional extra HR images (DIV2K/Flickr2K), (2) **fine-tuning on the provided `train_data` only**, at full 64x64 resolution.
3. **Extra data** is degraded with the *same* kernel as the provided data — the notebook measures which down-sampling kernel reproduces the provided LR images and uses it.
4. Augmentation: random crops + the 8 dihedral transforms (flips / 90° rotations).
5. Adam optimiser, cosine LR with warm-up, mixed precision, gradient clipping, **EMA of weights** (evaluated model), best-checkpoint selection on validation PSNR.
6. **Test-time self-ensemble (x8 dihedral)** — used only if it improves validation PSNR.

**Rule compliance (from the instructor):** convolutional autoencoder only · residual blocks / channel & spatial attention allowed · **no** transformers, **no** GAN, **no** diffusion · no pre-trained weights are used (trained from scratch) · augmentation and external datasets are used.

## 1. Setup

In [ ]:
!pip install -q sewar scipy matplotlib tqdm pillow

In [ ]:
import os, glob, math, time, copy, json, random, zipfile, gc
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from scipy.io import savemat
import re

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset, RandomSampler

from sewar.full_ref import psnr, ssim, rmse

try:
    import cv2
except ImportError:
    cv2 = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"           # fp16 mixed precision on GPU
torch.backends.cudnn.benchmark = True
print(f"Device: {device} | AMP: {USE_AMP} | torch {torch.__version__}")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

RESAMPLE_BICUBIC = getattr(Image, "Resampling", Image).BICUBIC
RESAMPLE_LANCZOS = getattr(Image, "Resampling", Image).LANCZOS

## 2. Configuration (everything you may want to change is here)
- `SMOKE_TEST = True` runs a tiny model for a few iterations so you can check the whole pipeline (data, training, evaluation, `submission.mat`) in a couple of minutes **before** launching the long run.
- `EXTRA_HR_DIRS`: folders with extra **high-resolution** images (e.g. Kaggle *DIV2K* `DIV2K_train_HR` and *Flickr2K*). Leave empty to train on the provided data only.
- `PRESET`: `"large"` (~12 M params, best accuracy, slowest) or `"medium"` (~3 M, ~3x faster).

In [ ]:
SMOKE_TEST = False

# ---------------- paths (None = auto-discover) ----------------
TRAIN_ROOT = "/kaggle/input/datasets/marionettist/fdsarew32"      # folder that contains LR/ and HR/   e.g. "/kaggle/input/datasets/marionettist/fdsarew32"
TEST_DIR   = "/kaggle/input/datasets/marionettist/test-lr"      # folder that contains 001.png..020.png e.g. "/kaggle/input/datasets/marionettist/test-lr"
EXTRA_HR_DIRS = ["/kaggle/input/datasets/anvu1204/df2kdata/DF2K_train_HR"]     # e.g. ["/kaggle/input/div2k-high-resolution-images/DIV2K_train_HR", "/kaggle/input/flickr2k/Flickr2K"]
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
SEARCH_ROOTS = ["/kaggle/working", "/kaggle/input", "/content", "."]

# ---------------- model ----------------
PRESETS = {
    "large":  dict(ch=(96, 128, 192), enc_blocks=(4, 4), bott_blocks=8, dec_blocks=(4, 4), head_ch=64, head_blocks=2),
    "medium": dict(ch=(64, 96, 128),  enc_blocks=(3, 3), bott_blocks=6, dec_blocks=(3, 3), head_ch=48, head_blocks=1),
    "tiny":   dict(ch=(16, 24, 32),   enc_blocks=(1, 1), bott_blocks=1, dec_blocks=(1, 1), head_ch=16, head_blocks=1),
}
PRESET = "medium"
USE_BN = False          # BatchNorm in encoder/decoder groups (allowed by the instructor, usually hurts PSNR)

# ---------------- data ----------------
VAL_FRAC = 0.05                 # share of the provided pairs kept for validation
BATCH_SIZE = 16
NUM_WORKERS = min(4, os.cpu_count() or 1)
PROVIDED_REPEAT = 2             # oversampling of the provided pairs in stage 1 (extra data is mixed in)
EXTRA_CROPS_PER_IMAGE = 8
MAX_EXTRA_PATCHES = 12000
EXTRA_SCALES = (1.0, 0.75, 0.5) # random pre-downscale of extra images (content-scale diversity, cleaner HR)
MATCH_MIN_PSNR = 40.0           # extra data is disabled if no down-sampling kernel reproduces the provided LR this well

# ---------------- training ----------------
STAGE1 = dict(name="stage1_joint",    iters=80_000, lr=3e-4, min_lr=1e-6, warmup=1000, loss="charbonnier", lr_patch=48)
STAGE2 = dict(name="stage2_finetune", iters=20_000, lr=1e-4, min_lr=1e-6, warmup=200,  loss="mse",          lr_patch=64)
EVAL_EVERY = 2000
EMA_DECAY = 0.999
GRAD_CLIP = 1.0

if SMOKE_TEST:
    PRESET = "tiny"
    STAGE1.update(iters=60, warmup=10)
    STAGE2.update(iters=30, warmup=5)
    EVAL_EVERY = 30
    NUM_WORKERS = 0
    EXTRA_HR_DIRS = []

print("Preset:", PRESET, PRESETS[PRESET])

## 3. Data: discovery, loading, train/validation split

In [ ]:
def extract_zip_if_needed(zip_name, out_dir):
    # Extract e.g. train_data.zip -> train_data/ if the zip is found and the folder does not exist yet.
    if os.path.isdir(out_dir):
        return
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for z in glob.glob(os.path.join(root, "**", zip_name), recursive=True):
            with zipfile.ZipFile(z) as zf:
                zf.extractall(out_dir)
            print(f"Extracted {z} -> {out_dir}/")
            return

def find_train_root():
    if TRAIN_ROOT and os.path.isdir(TRAIN_ROOT):
        return TRAIN_ROOT
    candidates = ["/kaggle/input/datasets/marionettist/fdsarew32"] + SEARCH_ROOTS
    for root in candidates:
        if not os.path.isdir(root):
            continue
        for lr_dir in glob.glob(os.path.join(root, "**", "LR"), recursive=True):
            parent = os.path.dirname(lr_dir)
            if os.path.isdir(os.path.join(parent, "HR")):
                return parent
    raise FileNotFoundError("Could not find a folder containing LR/ and HR/. Set TRAIN_ROOT manually.")

def load_pairs(root):
    # Pair LR and HR images by (parent folder, file name) so ordering can never be mismatched.
    def key_map(sub):
        out = {}
        for p in glob.glob(os.path.join(root, "**", sub, "*.png"), recursive=True):
            out[(os.path.dirname(os.path.dirname(p)), os.path.basename(p))] = p
        return out
    lr_map, hr_map = key_map("LR"), key_map("HR")
    keys = sorted(set(lr_map) & set(hr_map))
    if len(keys) == 0 or len(lr_map) != len(hr_map):
        raise RuntimeError(f"LR/HR mismatch: {len(lr_map)} LR files, {len(hr_map)} HR files, {len(keys)} matched.")
    lr_list, hr_list = [], []
    for k in tqdm(keys, desc="Loading train pairs"):
        lr_list.append(np.array(Image.open(lr_map[k]).convert("RGB"), dtype=np.uint8))
        hr_list.append(np.array(Image.open(hr_map[k]).convert("RGB"), dtype=np.uint8))
    lr = torch.from_numpy(np.stack(lr_list)).permute(0, 3, 1, 2).contiguous()
    hr = torch.from_numpy(np.stack(hr_list)).permute(0, 3, 1, 2).contiguous()
    return lr, hr

extract_zip_if_needed("train_data.zip", "train_data")
extract_zip_if_needed("test_lr.zip", "test_lr")

train_root = find_train_root()
print("Train root:", train_root)
LR_ALL, HR_ALL = load_pairs(train_root)
assert LR_ALL.shape[1:] == (3, 64, 64) and HR_ALL.shape[1:] == (3, 256, 256), (LR_ALL.shape, HR_ALL.shape)

N = len(LR_ALL)
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(N, generator=g)
n_val = int(min(max(32, VAL_FRAC * N), 0.2 * N))
val_idx, tr_idx = perm[:n_val], perm[n_val:]
VAL_LR, VAL_HR = LR_ALL[val_idx].clone(), HR_ALL[val_idx].clone()
TR_LR, TR_HR = LR_ALL[tr_idx].clone(), HR_ALL[tr_idx].clone()
del LR_ALL, HR_ALL
gc.collect()
print(f"Provided pairs: {N} | train {len(TR_LR)} | validation {len(VAL_LR)}")

### 3.1 Which down-sampling kernel produced the provided LR images?
We try several bicubic implementations on the provided HR images and compare with the provided LR images. The best match is reused to create LR images for any **extra** HR data, so that extra data follows exactly the same degradation as the test set.

In [ ]:
def to_hwc(t):
    return np.ascontiguousarray(t.permute(1, 2, 0).numpy())

def down_pil(hr):
    h, w = hr.shape[:2]
    return np.array(Image.fromarray(hr).resize((w // 4, h // 4), RESAMPLE_BICUBIC))

def _down_torch(hr, antialias):
    t = torch.from_numpy(hr).permute(2, 0, 1).float().unsqueeze(0)
    y = F.interpolate(t, size=(hr.shape[0] // 4, hr.shape[1] // 4), mode="bicubic",
                      align_corners=False, antialias=antialias)
    return y.round().clamp(0, 255).squeeze(0).permute(1, 2, 0).byte().numpy()

def down_torch_aa(hr):
    return _down_torch(hr, True)

def down_torch_plain(hr):
    return _down_torch(hr, False)

CANDIDATES = {
    "PIL bicubic (antialiased)": down_pil,
    "torch bicubic antialias=True": down_torch_aa,
    "torch bicubic antialias=False": down_torch_plain,
}
if cv2 is not None:
    CANDIDATES["OpenCV INTER_CUBIC"] = lambda hr: cv2.resize(hr, (hr.shape[1] // 4, hr.shape[0] // 4), interpolation=cv2.INTER_CUBIC)
    CANDIDATES["OpenCV INTER_AREA"] = lambda hr: cv2.resize(hr, (hr.shape[1] // 4, hr.shape[0] // 4), interpolation=cv2.INTER_AREA)

def np_psnr(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return 99.0 if mse < 1e-10 else 10 * math.log10(255.0 ** 2 / mse)

n_check = min(100, len(TR_LR))
scores = {}
for name, fn in CANDIDATES.items():
    vals = [np_psnr(fn(to_hwc(TR_HR[i])), to_hwc(TR_LR[i])) for i in range(n_check)]
    scores[name] = float(np.mean(vals))
    print(f"{name:35s} -> PSNR vs provided LR: {scores[name]:6.2f} dB")

BEST_KERNEL = max(scores, key=scores.get)
DOWN_FN = CANDIDATES[BEST_KERNEL]
print(f"\nBest matching kernel: {BEST_KERNEL} ({scores[BEST_KERNEL]:.2f} dB)")
KERNEL_MATCH_OK = scores[BEST_KERNEL] >= MATCH_MIN_PSNR
if not KERNEL_MATCH_OK:
    print(f"WARNING: best match < {MATCH_MIN_PSNR} dB -> the provided LR images were probably made with another pipeline "
          "(blur/noise/JPEG/different kernel). Extra data will be DISABLED to avoid a train/test mismatch.")

# quick visual sanity check of the provided pairs
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for j in range(4):
    axes[0, j].imshow(to_hwc(TR_LR[j])); axes[0, j].set_title("LR 64x64"); axes[0, j].axis("off")
    axes[1, j].imshow(to_hwc(TR_HR[j])); axes[1, j].set_title("HR 256x256"); axes[1, j].axis("off")
plt.tight_layout(); plt.show()

### 3.2 Optional extra training data (same degradation as the provided data)
Random 256x256 crops are taken from the extra HR images and down-sampled with the kernel found above. If `EXTRA_HR_DIRS` is empty, this cell does nothing.

In [ ]:
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".webp")

LOWRES_PAT = re.compile(r"(^|[^a-z])(lr|bicubic|unknown|x[2348])([^a-z]|$)")
VALID_PAT  = re.compile(r"(^|[^a-z])val(id|idation)?([^a-z]|$)")

def list_images(dirs):
    files = []
    for d in dirs:
        for dp, _, fn in os.walk(d):
            rel = os.path.relpath(dp, d).lower()
            if rel != "." and (LOWRES_PAT.search(rel) or VALID_PAT.search(rel)):
                continue          # skip LR_bicubic / LR_unknown / X2-X8 / valid sub-folders
            files += [os.path.join(dp, f) for f in fn if f.lower().endswith(IMG_EXT)]
    return sorted(set(files))

def build_extra_pairs(dirs, crops_per_img, max_patches, down_fn, scales, seed=SEED):
    files = list_images(dirs)
    print(f"Found {len(files)} extra images")
    rng = random.Random(seed)
    rng.shuffle(files)
    hr_list, lr_list = [], []
    for f in tqdm(files, desc="Building extra pairs"):
        if len(hr_list) >= max_patches:
            break
        try:
            img = Image.open(f).convert("RGB")
        except Exception:
            continue
        s = rng.choice(scales)
        if s != 1.0:
            img = img.resize((max(1, int(img.width * s)), max(1, int(img.height * s))), RESAMPLE_LANCZOS)
        if min(img.size) < 256:
            continue
        arr = np.asarray(img)
        H, W = arr.shape[:2]
        for _ in range(crops_per_img):
            t, l = rng.randint(0, H - 256), rng.randint(0, W - 256)
            crop = arr[t:t + 256, l:l + 256]
            if crop.std() < 6:          # skip almost flat patches
                continue
            crop = np.ascontiguousarray(crop)
            hr_list.append(crop)
            lr_list.append(down_fn(crop))
    if len(hr_list) == 0:
        return None, None
    lr = torch.from_numpy(np.stack(lr_list)).permute(0, 3, 1, 2).contiguous()
    hr = torch.from_numpy(np.stack(hr_list)).permute(0, 3, 1, 2).contiguous()
    return lr, hr

EXTRA_LR, EXTRA_HR = None, None
if EXTRA_HR_DIRS and KERNEL_MATCH_OK:
    EXTRA_LR, EXTRA_HR = build_extra_pairs(EXTRA_HR_DIRS, EXTRA_CROPS_PER_IMAGE, MAX_EXTRA_PATCHES, DOWN_FN, EXTRA_SCALES)
    if EXTRA_LR is not None:
        print(f"Extra pairs: {len(EXTRA_LR)} | LR {tuple(EXTRA_LR.shape[1:])} | HR {tuple(EXTRA_HR.shape[1:])}")
else:
    print("No extra data used.")
    if os.path.isdir("/kaggle/input"):
        print("Folders available in /kaggle/input (to fill EXTRA_HR_DIRS):")
        for d in sorted(glob.glob("/kaggle/input/*"))[:20]:
            print("  ", d)

### 3.3 Datasets with augmentation (random crop + 8 dihedral transforms)

In [ ]:
class PairDataset(Dataset):
    # In-memory uint8 pairs. A random LR crop (lr_patch x lr_patch) and the matching HR crop are taken,
    # then the same random flip / 90-degree rotation is applied to both.
    def __init__(self, lr_u8, hr_u8, lr_patch=None, augment=True, scale=4):
        assert len(lr_u8) == len(hr_u8)
        self.lr, self.hr = lr_u8, hr_u8
        self.patch, self.augment, self.s = lr_patch, augment, scale

    def __len__(self):
        return len(self.lr)

    def __getitem__(self, i):
        lr, hr, s = self.lr[i], self.hr[i], self.s
        if self.patch is not None and self.patch < lr.shape[-1]:
            p = self.patch
            t = random.randint(0, lr.shape[-2] - p)
            l = random.randint(0, lr.shape[-1] - p)
            lr = lr[:, t:t + p, l:l + p]
            hr = hr[:, t * s:(t + p) * s, l * s:(l + p) * s]
        if self.augment:
            if random.random() < 0.5:
                lr, hr = lr.flip(-1), hr.flip(-1)
            k = random.randint(0, 3)
            if k:
                lr, hr = torch.rot90(lr, k, dims=(-2, -1)), torch.rot90(hr, k, dims=(-2, -1))
        return lr.contiguous().float().div_(255.0), hr.contiguous().float().div_(255.0)

def make_train_dataset(lr_patch, include_extra):
    parts = [PairDataset(TR_LR, TR_HR, lr_patch) for _ in range(PROVIDED_REPEAT if include_extra else 1)]
    if include_extra and EXTRA_LR is not None:
        parts.append(PairDataset(EXTRA_LR, EXTRA_HR, lr_patch))
    return ConcatDataset(parts)

## 4. Model — Residual-Attention U-Net Convolutional Autoencoder
```
LR 64x64 ──► stem ──► [Enc1 @64] ──► down ──► [Enc2 @32] ──► down ──► [Bottleneck @16, CA+SA]   (encoder -> latent)
                         │ skip            │ skip                              │
                         ▼                 ▼                                   ▼
                  [Dec1 @64] ◄─ PS-up ◄ [Dec2 @32] ◄─ PS-up ◄──────────────────┘                (decoder)
                         + stem (global skip)
                         ▼
          PS-up x2 (64->128) ─► refine group ─► PS-up x2 (128->256) ─► conv ─► conv(3)  = residual
 output = bicubic_x4(LR) + residual                                                         (x4 head)
```

In [ ]:
def conv3x3(i, o, stride=1):
    return nn.Conv2d(i, o, 3, stride=stride, padding=1)

def icnr_(conv, scale=2):
    # ICNR init: sub-pixel conv starts as nearest-neighbour upsampling -> no checkerboard artefacts.
    out_ch, in_ch, kh, kw = conv.weight.shape
    sub = torch.zeros(out_ch // (scale ** 2), in_ch, kh, kw)
    nn.init.kaiming_normal_(sub, nonlinearity="relu")
    conv.weight.data.copy_(sub.repeat_interleave(scale ** 2, dim=0))
    nn.init.zeros_(conv.bias)

class ChannelAttention(nn.Module):
    def __init__(self, ch, reduction=16):
        super().__init__()
        mid = max(ch // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Conv2d(ch, mid, 1), nn.ReLU(inplace=True), nn.Conv2d(mid, ch, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.fc(self.pool(x))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k // 2)

    def forward(self, x):
        pooled = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        return x * torch.sigmoid(self.conv(pooled))

class ResAttnBlock(nn.Module):
    # conv-ReLU-conv -> channel attention (-> spatial attention) -> scaled residual connection
    def __init__(self, ch, use_sa=False, use_bn=False, res_scale=0.2, reduction=16):
        super().__init__()
        layers = [conv3x3(ch, ch)]
        if use_bn:
            layers.append(nn.BatchNorm2d(ch))
        layers.append(nn.ReLU(inplace=True))
        layers.append(conv3x3(ch, ch))
        if use_bn:
            layers.append(nn.BatchNorm2d(ch))
        self.body = nn.Sequential(*layers)
        self.ca = ChannelAttention(ch, reduction)
        self.sa = SpatialAttention() if use_sa else nn.Identity()
        self.res_scale = res_scale

    def forward(self, x):
        return x + self.res_scale * self.sa(self.ca(self.body(x)))

class ResGroup(nn.Module):
    def __init__(self, ch, n_blocks, **kw):
        super().__init__()
        self.body = nn.Sequential(*[ResAttnBlock(ch, **kw) for _ in range(n_blocks)], conv3x3(ch, ch))

    def forward(self, x):
        return x + self.body(x)

class PixelShuffleUp(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = conv3x3(in_ch, out_ch * 4)
        icnr_(self.conv, 2)
        self.ps = nn.PixelShuffle(2)

    def forward(self, x):
        return self.ps(self.conv(x))

class SRAutoencoder(nn.Module):
    def __init__(self, ch=(96, 128, 192), enc_blocks=(4, 4), bott_blocks=8, dec_blocks=(4, 4),
                 head_ch=64, head_blocks=2, use_bn=False, scale=4):
        super().__init__()
        assert scale == 4
        c1, c2, c3 = ch
        self.scale = scale
        # ---- encoder ----
        self.stem = conv3x3(3, c1)
        self.enc1 = ResGroup(c1, enc_blocks[0], use_bn=use_bn)                          # @64x64
        self.down1 = conv3x3(c1, c2, stride=2)                                          # 64 -> 32
        self.enc2 = ResGroup(c2, enc_blocks[1], use_bn=use_bn)                          # @32x32
        self.down2 = conv3x3(c2, c3, stride=2)                                          # 32 -> 16
        # ---- bottleneck (latent code) ----
        self.bottleneck = ResGroup(c3, bott_blocks, use_sa=True, use_bn=use_bn)         # @16x16
        # ---- decoder (with skip connections) ----
        self.up2 = PixelShuffleUp(c3, c2)                                               # 16 -> 32
        self.fuse2 = nn.Conv2d(2 * c2, c2, 1)
        self.dec2 = ResGroup(c2, dec_blocks[1], use_sa=True, use_bn=use_bn)
        self.up1 = PixelShuffleUp(c2, c1)                                               # 32 -> 64
        self.fuse1 = nn.Conv2d(2 * c1, c1, 1)
        self.dec1 = ResGroup(c1, dec_blocks[0], use_bn=use_bn)
        # ---- x4 reconstruction head ----
        final = nn.Conv2d(head_ch, 3, 3, padding=1)
        nn.init.zeros_(final.weight)
        nn.init.zeros_(final.bias)                                                      # start exactly at bicubic
        self.head = nn.Sequential(
            PixelShuffleUp(c1, head_ch),                                                # 64 -> 128
            ResGroup(head_ch, head_blocks),
            PixelShuffleUp(head_ch, head_ch),                                           # 128 -> 256
            conv3x3(head_ch, head_ch), nn.LeakyReLU(0.1, inplace=True),
            final,
        )

    def forward(self, x):
        assert x.shape[-1] % 4 == 0 and x.shape[-2] % 4 == 0, "input size must be a multiple of 4"
        base = F.interpolate(x.float(), scale_factor=self.scale, mode="bicubic", align_corners=False)
        s0 = self.stem(x)
        e1 = self.enc1(s0)
        e2 = self.enc2(self.down1(e1))
        z = self.bottleneck(self.down2(e2))
        d2 = self.dec2(self.fuse2(torch.cat([self.up2(z), e2], dim=1)))
        d1 = self.dec1(self.fuse1(torch.cat([self.up1(d2), e1], dim=1)))
        res = self.head(d1 + s0)
        return base + res.float()

def build_model():
    return SRAutoencoder(use_bn=USE_BN, **PRESETS[PRESET]).to(device)

model = build_model()
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Preset '{PRESET}' | trainable parameters: {n_params:,}")
with torch.no_grad():
    _y = model(torch.zeros(1, 3, 64, 64, device=device))
assert _y.shape == (1, 3, 256, 256), _y.shape
print("Dry-run OK:", tuple(_y.shape))
del _y

## 5. Losses, EMA, metrics and test-time augmentation

In [ ]:
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps

    def forward(self, x, y):
        return torch.mean(torch.sqrt((x - y) ** 2 + self.eps ** 2))

def make_loss(name):
    if name == "charbonnier":
        return CharbonnierLoss()
    if name == "l1":
        return nn.L1Loss()
    if name == "mse":
        return nn.MSELoss()
    raise ValueError(name)

class EMA:
    # Exponential moving average of the weights; the EMA copy is the model that gets evaluated / submitted.
    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay
        self.steps = 0

    @torch.no_grad()
    def update(self, model):
        self.steps += 1
        d = min(self.decay, (1 + self.steps) / (10 + self.steps))
        src = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(src[k].detach(), alpha=1.0 - d)
            else:
                v.copy_(src[k])

def tensor_to_uint8_hwc(tensor):
    img_np = tensor.detach().cpu().float().numpy().transpose(1, 2, 0)
    return np.clip(img_np * 255.0, 0, 255).round().astype(np.uint8)

@torch.no_grad()
def psnr_uint8(pred, gt):
    # Per-image PSNR computed on 8-bit rounded images, identical to sewar.psnr on uint8 arrays.
    p = (pred.clamp(0, 1) * 255.0).round()
    t = (gt * 255.0).round()
    mse = ((p - t) ** 2).flatten(1).mean(1)
    return 10.0 * torch.log10(255.0 ** 2 / mse.clamp_min(1e-10))

@torch.no_grad()
def predict_batch(net, x, tta=False, amp=False):
    # x: float tensor in [0,1] on `device`. With tta=True the 8 dihedral variants are averaged (self-ensemble).
    def run(inp):
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(amp and USE_AMP)):
            return net(inp).float()
    if not tta:
        return run(x).clamp(0, 1)
    out = 0
    for flip in (False, True):
        for k in range(4):
            xi = x.flip(-1) if flip else x
            xi = torch.rot90(xi, k, dims=(-2, -1))
            yi = torch.rot90(run(xi), -k, dims=(-2, -1))
            out = out + (yi.flip(-1) if flip else yi)
    return (out / 8.0).clamp(0, 1)

@torch.no_grad()
def evaluate(net, lr_u8, hr_u8, criterion=None, tta=False, amp=True, bs=16):
    net.eval()
    psnrs, loss_sum = [], 0.0
    for i in range(0, len(lr_u8), bs):
        x = lr_u8[i:i + bs].to(device).float() / 255.0
        y = hr_u8[i:i + bs].to(device).float() / 255.0
        p = predict_batch(net, x, tta=tta, amp=amp)
        psnrs.append(psnr_uint8(p, y).cpu())
        if criterion is not None:
            loss_sum += criterion(p, y).item() * len(x)
    return float(torch.cat(psnrs).mean()), loss_sum / len(lr_u8)

# Reference: plain bicubic interpolation on the validation split
@torch.no_grad()
def bicubic_val_psnr():
    vals = []
    for i in range(0, len(VAL_LR), 16):
        x = VAL_LR[i:i + 16].to(device).float() / 255.0
        y = VAL_HR[i:i + 16].to(device).float() / 255.0
        p = F.interpolate(x, scale_factor=4, mode="bicubic", align_corners=False).clamp(0, 1)
        vals.append(psnr_uint8(p, y).cpu())
    return float(torch.cat(vals).mean())

BICUBIC_VAL_PSNR = bicubic_val_psnr()
print(f"Bicubic baseline on validation split: {BICUBIC_VAL_PSNR:.3f} dB")

## 6. Training routine (cosine LR + warm-up, AMP, EMA, best-checkpoint on validation PSNR)

In [ ]:
HISTORY = []          # one record per evaluation, across both stages
ITER_OFFSET = 0

def lr_at(it, total, base, minimum, warmup):
    if it <= warmup:
        return base * it / max(1, warmup)
    p = (it - warmup) / max(1, total - warmup)
    return minimum + 0.5 * (base - minimum) * (1.0 + math.cos(math.pi * p))

def train_stage(cfg, net, train_ds, init_ckpt=None):
    global ITER_OFFSET
    name, total = cfg["name"], cfg["iters"]
    save_path = os.path.join(OUT_DIR, f"{name}_best_ema.pt")
    if init_ckpt is not None:
        net.load_state_dict(torch.load(init_ckpt, map_location=device))
        print(f"[{name}] initialised from {init_ckpt}")

    criterion = make_loss(cfg["loss"])
    optimizer = optim.Adam(net.parameters(), lr=cfg["lr"], betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)
    ema = EMA(net, EMA_DECAY)

    gen = torch.Generator().manual_seed(SEED)
    sampler = RandomSampler(train_ds, replacement=True, num_samples=total * BATCH_SIZE, generator=gen)
    loader_kw = dict(batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS,
                     pin_memory=(device.type == "cuda"), drop_last=True)
    if NUM_WORKERS > 0:
        loader_kw.update(persistent_workers=True, prefetch_factor=4)
    loader = DataLoader(train_ds, **loader_kw)

    print(f"[{name}] {total} iters | batch {BATCH_SIZE} | LR patch {cfg['lr_patch']} | loss {cfg['loss']} | "
          f"train samples/pool {len(train_ds)}")
    best_psnr, run_loss, run_n, t0 = -1.0, 0.0, 0, time.time()
    net.train()
    pbar = tqdm(total=total, desc=name)
    for it, (lr_b, hr_b) in enumerate(loader, start=1):
        cur_lr = lr_at(it, total, cfg["lr"], cfg["min_lr"], cfg["warmup"])
        for grp in optimizer.param_groups:
            grp["lr"] = cur_lr
        lr_b = lr_b.to(device, non_blocking=True)
        hr_b = hr_b.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            pred = net(lr_b)
        loss = criterion(pred.float(), hr_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        ema.update(net)

        run_loss += loss.item() * lr_b.size(0)
        run_n += lr_b.size(0)
        pbar.update(1)

        if it % EVAL_EVERY == 0 or it == total:
            val_psnr, val_loss = evaluate(ema.module, VAL_LR, VAL_HR, criterion=criterion, amp=True)
            tr_loss = run_loss / max(1, run_n)
            run_loss, run_n = 0.0, 0
            HISTORY.append(dict(stage=name, iter=ITER_OFFSET + it, lr=cur_lr, train_loss=tr_loss,
                                val_loss=val_loss, val_psnr=val_psnr))
            if val_psnr > best_psnr:
                best_psnr = val_psnr
                torch.save(ema.module.state_dict(), save_path)
            speed = it / (time.time() - t0)
            print(f"[{name}] it {it}/{total} | lr {cur_lr:.2e} | train {tr_loss:.5f} | val {val_loss:.5f} | "
                  f"val PSNR {val_psnr:.3f} dB | best {best_psnr:.3f} | {speed:.2f} it/s")
            net.train()
    pbar.close()
    ITER_OFFSET += total
    net.load_state_dict(torch.load(save_path, map_location=device))
    return dict(name=name, best_psnr=best_psnr, best_path=save_path)

## 7. Run training
**Stage 1** trains on the provided data (oversampled x`PROVIDED_REPEAT`) + extra data, with random crops and Charbonnier loss.
**Stage 2** starts from the best stage-1 weights and fine-tunes on the **provided `train_data` only** (full 64x64 images, MSE loss, small LR) so the model specialises on the test distribution.

In [ ]:
RESULTS = []
model = build_model()
ds1 = make_train_dataset(STAGE1["lr_patch"], include_extra=True)
RESULTS.append(train_stage(STAGE1, model, ds1))
print(f"\nStage 1 best val PSNR: {RESULTS[-1]['best_psnr']:.3f} dB (bicubic: {BICUBIC_VAL_PSNR:.3f} dB)")

In [ ]:
ds2 = make_train_dataset(STAGE2["lr_patch"], include_extra=False)
RESULTS.append(train_stage(STAGE2, model, ds2, init_ckpt=RESULTS[0]["best_path"]))
print(f"\nStage 2 best val PSNR: {RESULTS[-1]['best_psnr']:.3f} dB")
with open(os.path.join(OUT_DIR, "training_history.json"), "w") as f:
    json.dump(HISTORY, f, indent=1)

## 8. Training / validation curves

In [ ]:
iters = [h["iter"] for h in HISTORY]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(iters, [h["train_loss"] for h in HISTORY], marker="o", label="Training loss (raw model, random crops)")
axes[0].plot(iters, [h["val_loss"] for h in HISTORY], marker="s", label="Validation loss (EMA model, full images)")
axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Loss"); axes[0].set_yscale("log")
axes[0].set_title("Training and validation loss"); axes[0].grid(True, ls="--", alpha=0.5); axes[0].legend()
axes[1].plot(iters, [h["val_psnr"] for h in HISTORY], marker="o", color="tab:green", label="Validation PSNR (EMA)")
axes[1].axhline(BICUBIC_VAL_PSNR, color="gray", ls="--", label=f"Bicubic ({BICUBIC_VAL_PSNR:.2f} dB)")
axes[1].set_xlabel("Iteration"); axes[1].set_ylabel("PSNR (dB)")
axes[1].set_title("Validation PSNR"); axes[1].grid(True, ls="--", alpha=0.5); axes[1].legend()
boundary = STAGE1["iters"]
for ax in axes:
    ax.axvline(boundary, color="red", ls=":", alpha=0.6)
    ax.text(boundary, ax.get_ylim()[1], " stage 2 (fine-tune)", color="red", va="top", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150)
plt.show()

## 9. Final validation with `sewar` (PSNR / SSIM / RMSE) and visual comparison

In [ ]:
best = max(RESULTS, key=lambda r: r["best_psnr"])
print("Selected checkpoint:", best["name"], f"({best['best_psnr']:.3f} dB)")
final_model = build_model()
final_model.load_state_dict(torch.load(best["best_path"], map_location=device))
final_model.eval()

# Is the x8 self-ensemble helpful? (fp32, full validation split)
psnr_plain, _ = evaluate(final_model, VAL_LR, VAL_HR, tta=False, amp=False)
psnr_tta, _ = evaluate(final_model, VAL_LR, VAL_HR, tta=True, amp=False)
USE_TTA = psnr_tta >= psnr_plain
print(f"Val PSNR without TTA: {psnr_plain:.4f} dB | with x8 TTA: {psnr_tta:.4f} dB -> use TTA: {USE_TTA}")

def predict_uint8(net, lr_t):
    x = lr_t.unsqueeze(0).to(device).float()          # expects values in [0, 1]
    return tensor_to_uint8_hwc(predict_batch(net, x, tta=USE_TTA, amp=False).squeeze(0))

n_eval = min(50, len(VAL_LR))
rows = {"Bicubic": ([], [], []), "RA-UNet-AE (ours)": ([], [], [])}
for i in tqdm(range(n_eval), desc="sewar metrics"):
    gt = to_hwc(VAL_HR[i])
    lr_f = VAL_LR[i].unsqueeze(0).float() / 255.0
    bic = tensor_to_uint8_hwc(F.interpolate(lr_f, scale_factor=4, mode="bicubic", align_corners=False).clamp(0, 1).squeeze(0))
    ours = predict_uint8(final_model, VAL_LR[i].float() / 255.0)
    for key, pred in (("Bicubic", bic), ("RA-UNet-AE (ours)", ours)):
        rows[key][0].append(float(psnr(gt, pred)))
        rows[key][1].append(float(ssim(gt, pred)[0]))
        rows[key][2].append(float(rmse(gt, pred)))

print("=" * 66)
print(f"{'Method':22s}{'PSNR (dB) up':>14s}{'SSIM up':>12s}{'RMSE down':>12s}")
print("-" * 66)
for key, (p, s, r) in rows.items():
    print(f"{key:22s}{np.mean(p):14.4f}{np.mean(s):12.4f}{np.mean(r):12.4f}")
print("=" * 66)

fig, axes = plt.subplots(3, 4, figsize=(15, 11))
for r in range(3):
    gt = to_hwc(VAL_HR[r])
    lr_f = VAL_LR[r].unsqueeze(0).float() / 255.0
    bic = tensor_to_uint8_hwc(F.interpolate(lr_f, scale_factor=4, mode="bicubic", align_corners=False).clamp(0, 1).squeeze(0))
    ours = predict_uint8(final_model, VAL_LR[r].float() / 255.0)
    for c, (img, title) in enumerate([(to_hwc(VAL_LR[r]), "LR input (64x64)"),
                                      (bic, f"Bicubic ({psnr(gt, bic):.2f} dB)"),
                                      (ours, f"Ours ({psnr(gt, ours):.2f} dB)"),
                                      (gt, "Ground truth (256x256)")]):
        axes[r, c].imshow(img); axes[r, c].set_title(title); axes[r, c].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "visual_comparison.png"), dpi=150)
plt.show()

## 10. Generate `submission.mat` (20 test images, shape `(20, 256, 256, 3)`, `uint8`)

In [ ]:
def find_test_dir():
    cands = [TEST_DIR, "/kaggle/input/datasets/marionettist/test-lr", "test_lr"]
    for c in cands:
        if c and os.path.isdir(c) and glob.glob(os.path.join(c, "**", "001.png"), recursive=True):
            return c
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for p in glob.glob(os.path.join(root, "**", "001.png"), recursive=True):
            if Image.open(p).size == (64, 64):
                return os.path.dirname(p)
    raise FileNotFoundError("Could not find test images 001.png..020.png. Set TEST_DIR manually.")

def find_test_lr_image(index, root_folder):
    fname = f"{index:03d}.png"
    direct = os.path.join(root_folder, fname)
    if os.path.exists(direct):
        return direct
    matches = glob.glob(os.path.join(root_folder, "**", fname), recursive=True)
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Could not find '{fname}' inside '{root_folder}'.")

test_dir = find_test_dir()
print("Test folder:", test_dir)
final_model.eval()
predictions_list, sanity = [], []
for i in range(1, 21):
    lr_img = np.array(Image.open(find_test_lr_image(i, test_dir)).convert("RGB"), dtype=np.float32) / 255.0
    assert lr_img.shape == (64, 64, 3), f"Unexpected test image size for {i:03d}.png: {lr_img.shape}"
    lr_t = torch.from_numpy(lr_img.transpose(2, 0, 1))
    pred = predict_uint8(final_model, lr_t)
    predictions_list.append(pred)
    bic = tensor_to_uint8_hwc(F.interpolate(lr_t.unsqueeze(0), scale_factor=4, mode="bicubic", align_corners=False).clamp(0, 1).squeeze(0))
    sanity.append(float(psnr(bic, pred)))

predictions_array = np.stack(predictions_list, axis=0)
assert predictions_array.shape == (20, 256, 256, 3), f"Invalid shape: {predictions_array.shape}"
assert predictions_array.dtype == np.uint8, f"Invalid dtype: {predictions_array.dtype}"

sub_path = os.path.join(OUT_DIR, "submission.mat")
savemat(sub_path, {"predictions": predictions_array}, do_compression=True)
size_mb = os.path.getsize(sub_path) / (1024 * 1024)
print(f"SUCCESS! Saved '{sub_path}' ({size_mb:.2f} MB) | key 'predictions' | shape {predictions_array.shape} | dtype {predictions_array.dtype}")
print(f"Sanity check - mean PSNR between our outputs and plain bicubic upscaling of the test inputs: {np.mean(sanity):.2f} dB "
      "(should be clearly above ~25 dB; a very low value means something is wrong)")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for j in range(4):
    axes[0, j].imshow(np.array(Image.open(find_test_lr_image(j + 1, test_dir)).convert("RGB"))); axes[0, j].set_title(f"Test {j+1:03d} LR"); axes[0, j].axis("off")
    axes[1, j].imshow(predictions_array[j]); axes[1, j].set_title("Prediction (256x256)"); axes[1, j].axis("off")
plt.tight_layout(); plt.show()

summary = dict(preset=PRESET, params=n_params, use_bn=USE_BN, kernel=BEST_KERNEL, extra_pairs=(0 if EXTRA_LR is None else len(EXTRA_LR)),
               bicubic_val_psnr=BICUBIC_VAL_PSNR, best_stage=best["name"], best_val_psnr_ema=best["best_psnr"],
               val_psnr_tta=psnr_tta, val_psnr_no_tta=psnr_plain, used_tta=USE_TTA,
               sewar_val={k: dict(psnr=float(np.mean(v[0])), ssim=float(np.mean(v[1])), rmse=float(np.mean(v[2]))) for k, v in rows.items()})
with open(os.path.join(OUT_DIR, "report_summary.json"), "w") as f:
    json.dump(summary, f, indent=1)
print(json.dumps(summary, indent=1))